# Content-Based Movie Recommendation System
This notebook covers:
1. Data Cleaning & Preprocessing
2. Vectorization
3. Recommendation Function
4. Model Saving


In [ ]:
import pandas as pd
import ast
import re
import pickle
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 🔹 STEP 1: Data Cleaning & Preprocessing


In [ ]:
print("Loading datasets...")
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

print("Merging datasets...")
movies = movies.merge(credits, on="title")

# Select relevant columns
print("Selecting columns...")
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast']]
movies.dropna(inplace=True)


In [ ]:
# Helper function to parse JSON-like columns
def convert(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            L.append(i['name'])
    except Exception:
        pass
    return L

# For cast, limit to top 3 actors
def convert3(text):
    L = []
    try:
        counter = 0
        for i in ast.literal_eval(text):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
    except Exception:
        pass
    return L

print("Parsing JSON-like columns...")
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)

# For overview, split into a list
movies['overview'] = movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else [])

# Remove spaces between words in lists so that "Science Fiction" becomes "ScienceFiction"
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])

# Combine into tags
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast']

new_df = movies[['id', 'title', 'tags']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [ ]:
ps = PorterStemmer()

def stem_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

print("Applying stemming process...")
new_df['tags'] = new_df['tags'].apply(stem_text)

print("\n--- Cleaned DataFrame Head ---")
display(new_df.head())
print("\n--- Final Cleaned DataFrame Shape ---")
print(new_df.shape)


## 🔹 STEP 2: Vectorization


In [ ]:
print("\nVectorizing tags...")
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()

print("Computing cosine similarity...")
similarity = cosine_similarity(vectors)

print("\n--- Similarity Matrix Shape ---")
print(similarity.shape)

print('''
Why Cosine Similarity is appropriate here:
In this high-dimensional sparse vector space created by CountVectorizer, we are interested in the angle between movie vectors rather than their magnitude. 
Cosine similarity measures the orientation between vectors (how similar the tags are irrespective of word count or document length). This provides a normalized distance score between 0 and 1, where 1 means identical content, making it highly effective for text-based recommendations.
''')


## 🔹 STEP 3: Recommendation Function


In [ ]:
def recommend(movie_title):
    try:
        # Find index of movie
        movie_index = new_df[new_df['title'] == movie_title].index[0]
        # Retrieve similarity scores
        distances = similarity[movie_index]
        # Sort in descending order and exclude the queried movie itself (which is at index 0)
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        print(f"Top 5 Recommendations for '{movie_title}':")
        for i in movies_list:
            print(new_df.iloc[i[0]].title)
    except IndexError:
        print(f"Movie '{movie_title}' not found in the dataset.")

print("\n--- Example Usage ---")
recommend("Avatar")


## 🔹 STEP 4: Model Saving


In [ ]:
import pickle

print("\nSaving vectorizer and similarity matrix to pickle files...")
pickle.dump(cv, open('vectorizer.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

print("\nPipeline completed successfully!")


## 🔹 STEP 5: Streamlit App
Run the following cell to create `app.py`. After that, you can run the Streamlit app from your terminal using:
`streamlit run app.py`


In [ ]:
%%writefile app.py
import streamlit as st
import pickle
import pandas as pd
import requests

st.set_page_config(page_title="Movie Recommender", layout="wide")
st.title("🎬 Movie Recommendation System")

@st.cache_data
def load_data():
    movies_dict = pickle.load(open('movie_dict.pkl', 'rb'))
    movies = pd.DataFrame(movies_dict)
    similarity = pickle.load(open('similarity.pkl', 'rb'))
    return movies, similarity

movies, similarity = load_data()

def fetch_poster(movie_id):
    api_key = "8265bd1679663a7ea12ac168da84d2e8"
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&language=en-US"
    try:
        response = requests.get(url)
        data = response.json()
        poster_path = data.get('poster_path')
        if poster_path:
            return "https://image.tmdb.org/t/p/w500/" + poster_path
    except Exception as e:
        pass
    return "https://via.placeholder.com/500x750?text=No+Poster"

def recommend(movie_title):
    try:
        movie_index = movies[movies['title'] == movie_title].index[0]
        distances = similarity[movie_index]
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        recommended_movies = []
        recommended_movies_posters = []
        
        for i in movies_list:
            movie_id = movies.iloc[i[0]].id
            recommended_movies.append(movies.iloc[i[0]].title)
            recommended_movies_posters.append(fetch_poster(movie_id))
            
        return recommended_movies, recommended_movies_posters
    except IndexError:
        return [], []

st.markdown("### Find movies similar to your favorites")
selected_movie_name = st.selectbox('Type or select a movie from the dropdown', movies['title'].values)

if st.button('Recommend'):
    with st.spinner('Finding recommendations...'):
        names, posters = recommend(selected_movie_name)
        if names:
            cols = st.columns(5)
            for i, col in enumerate(cols):
                with col:
                    st.markdown(f"**{names[i]}**")
                    st.image(posters[i], use_container_width=True)
        else:
            st.error("Movie not found or no recommendations available.")
